# 🐼 Pandas: Zero to Master — A Guided Lab

Pandas is the tool for real tabular data in Python — think of it as a programmable
spreadsheet + SQL database. It's built directly on NumPy (which you just learned), so
everything transfers.

**How this lab works** — same as the NumPy lab:
- 📖 Theory → 🔬 Worked examples → ✏️ Your Turn → ✅ Solutions in every chapter.
- Run every cell. Attempt each exercise before revealing its solution.
- 🧠 mental models · ⚡ pro tips · ⚠️ traps.

**We learn on real, messy data** — a retail sales dataset with missing values, duplicates,
and inconsistent text (just like real life). You'll clean it and analyze it as you learn.

**What you'll master**
1. Series & DataFrame — the two core objects
2. Loading data & first-look inspection
3. Selecting data: `[]`, `.loc`, `.iloc`
4. Filtering rows (boolean & `.query`)
5. Handling missing data
6. Cleaning: duplicates, text, types
7. Creating & transforming columns
8. Sorting & ranking
9. GroupBy & aggregation (the heart of analysis)
10. Merging & joining tables
11. Datetime & time-series resampling
12. Pivot tables & reshaping
13. 🏆 Capstone: a full business analysis


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)

---
## Chapter 1 — Series & DataFrame

📖 **Theory.** Two objects power everything:
- A **Series** is a 1D labeled array (one column, with an index).
- A **DataFrame** is a 2D labeled table (many Series sharing one index) — rows and named columns.

🧠 **Mental model.** A DataFrame is a dict of Series. Each column is a Series; they all share
the same row index.

In [ ]:
# A Series -- values plus an index
s = pd.Series([10, 20, 30, 40], index=["a", "b", "c", "d"], name="sales")
print(s)
print("\nvalue at 'b':", s["b"])

# A DataFrame -- built from a dict of columns
df = pd.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard"],
    "price": [900, 25, 45],
    "in_stock": [True, False, True],
})
print("\n", df)
print("\ndtypes:\n", df.dtypes)

### ✏️ Your Turn 1.1
Build a DataFrame `students` with three columns: `name` (3 names), `age` (3 ages), and
`grade` (3 letter grades). Then print just the `name` column (which is a Series).

In [ ]:
students = None
# print the name column


✅ **Solution**
```python
students = pd.DataFrame({
    "name": ["Ana","Ben","Cara"],
    "age": [20, 22, 21],
    "grade": ["A","B","A"],
})
print(students["name"])
```

---
## Chapter 2 — Loading Data & First Look

📖 **Theory.** Real data comes from files. `pd.read_csv(path)` is the workhorse. The moment
you load data, you *always* run the same inspection ritual to understand what you have:

| Method | Tells you |
|---|---|
| `.head(n)` / `.tail(n)` | first/last n rows |
| `.shape` | (rows, columns) |
| `.info()` | column names, dtypes, non-null counts |
| `.describe()` | summary stats for numeric columns |
| `.columns` / `.dtypes` | column names / types |
| `.isna().sum()` | missing values per column |

⚡ **Pro tip.** Doing this ritual first prevents 90% of downstream bugs.

In [ ]:
df = pd.read_csv("datasets/retail_sales.csv", parse_dates=["order_date"])
print("shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print("Numeric summary:")
print(df.describe())
print("\nMissing values per column:")
print(df.isna().sum())

⚠️ **Common trap.** Without `parse_dates=[...]`, date columns load as plain strings and
`.dt` operations won't work. Notice we passed `parse_dates=["order_date"]` above.

### ✏️ Your Turn 2.1
1. How many rows and columns does `df` have?
2. Which columns contain missing values, and how many each?
3. What is the average `revenue` across all orders? (hint: `df["revenue"].mean()`)

In [ ]:
n_rows, n_cols = None, None
missing = None
avg_revenue = None
print(n_rows, n_cols, avg_revenue)

✅ **Solution**
```python
n_rows, n_cols = df.shape
missing = df.isna().sum()
avg_revenue = df["revenue"].mean()
```

---
## Chapter 3 — Selecting Data: `[]`, `.loc`, `.iloc`

📖 **Theory.** Three ways to select — know when to use each:
- `df["col"]` → one column (Series); `df[["a","b"]]` → multiple columns (DataFrame)
- `.loc[rows, cols]` → select by **label** (row index & column names)
- `.iloc[rows, cols]` → select by **integer position** (like NumPy)

🧠 **Mental model.** `.loc` speaks *names*, `.iloc` speaks *positions*. When your index is
just 0,1,2… they look similar, but `.loc[0:2]` includes 2 (label slice) while `.iloc[0:2]`
excludes 2 (positional slice) — a classic gotcha.

In [ ]:
# single & multiple columns
print(df["product"].head(3))
print("\n", df[["product", "revenue"]].head(3))

# .iloc -- by position
print("\nFirst 2 rows, first 3 columns (.iloc):")
print(df.iloc[0:2, 0:3])

# .loc -- by label (here row labels are 0,1,2...; column labels are names)
print("\nRows 0-2, selected columns (.loc):")
print(df.loc[0:2, ["product", "revenue"]])

⚠️ **Common trap.** `.loc[0:2]` returns **3 rows** (0,1,2 — label slicing is inclusive)
while `.iloc[0:2]` returns **2 rows** (0,1 — positional slicing is exclusive, like Python).

### ✏️ Your Turn 3.1
1. Select just the `region` and `units` columns (first 5 rows)
2. Using `.iloc`, get the value in the 4th row, 2nd column
3. Using `.loc`, get rows with index 10 through 12, showing only `product` and `revenue`

In [ ]:
two_cols = None
cell = None
loc_slice = None
print(two_cols); print(cell); print(loc_slice)

✅ **Solution**
```python
two_cols = df[["region", "units"]].head(5)
cell = df.iloc[3, 1]
loc_slice = df.loc[10:12, ["product", "revenue"]]
```

---
## Chapter 4 — Filtering Rows

📖 **Theory.** Keep only rows matching a condition — the equivalent of SQL `WHERE`.
- Boolean mask: `df[df["revenue"] > 1000]`
- Combine with `&` `|` `~` (parenthesize each condition!)
- `.query("revenue > 1000 and region == 'North'")` — often more readable
- `.isin([...])` — match any of several values

In [ ]:
# single condition
big_orders = df[df["revenue"] > 2000]
print("orders over $2000:", len(big_orders))

# multiple conditions -- note the parentheses and & (not 'and')
north_electronics = df[(df["region"] == "North") & (df["category"] == "Electronics")]
print("North electronics orders:", len(north_electronics))

# .query() -- cleaner syntax
same_thing = df.query("region == 'North' and category == 'Electronics'")
print(".query gives same count:", len(same_thing))

# .isin()
furniture_or_electronics = df[df["category"].isin(["Furniture", "Electronics"])]
print("furniture or electronics:", len(furniture_or_electronics))

### ✏️ Your Turn 4.1
1. Find all orders where `units` is greater than 5
2. Find orders from `Alice` OR `Bob` (the `sales_rep` column) using `.isin`
3. Using `.query`, find `Laptop` orders with `revenue` over 3000

In [ ]:
many_units = None
alice_or_bob = None
big_laptops = None
print(len(many_units) if many_units is not None else None)

✅ **Solution**
```python
many_units = df[df["units"] > 5]
alice_or_bob = df[df["sales_rep"].isin(["Alice","Bob"])]
big_laptops = df.query("product == 'Laptop' and revenue > 3000")
```

---
## Chapter 5 — Handling Missing Data

📖 **Theory.** Missing values (`NaN`) are everywhere in real data. Your options:
- `.isna()` / `.notna()` — detect missing
- `.dropna(subset=[...])` — drop rows with missing values
- `.fillna(value)` — fill with a constant, mean, median, or forward/backward fill
- Deciding *which* strategy is a judgment call, not automatic.

🧠 **Mental model.** Dropping loses data; filling invents data. Choose based on how much is
missing and whether a reasonable fill exists.

In [ ]:
print("missing before:\n", df[["units","region"]].isna().sum())

# Strategy 1: drop rows missing 'units'
df_dropped = df.dropna(subset=["units"]).copy()
print("\nrows after dropping missing units:", len(df_dropped))

# Strategy 2: fill missing 'units' with the median (a common numeric choice)
df_filled = df.copy()
median_units = df_filled["units"].median()
df_filled["units"] = df_filled["units"].fillna(median_units)
print("missing units after fill:", df_filled["units"].isna().sum())

# Strategy 3: fill missing categorical 'region' with a placeholder
df_filled["region"] = df_filled["region"].fillna("Unknown")
print("missing region after fill:", df_filled["region"].isna().sum())

⚡ **Pro tip.** For numeric columns, median is often safer than mean (robust to outliers).
For categoricals, a placeholder like `"Unknown"` keeps the row instead of dropping it.

### ✏️ Your Turn 5.1
Working on a fresh copy `work = df.copy()`:
1. How many rows have a missing `region`?
2. Fill missing `units` with the **mean** units (rounded to a whole number)
3. Drop any remaining rows that still have missing values anywhere

In [ ]:
work = df.copy()
missing_region = None
# fill units with mean...
# drop remaining missing...
final_rows = None
print(missing_region, final_rows)

✅ **Solution**
```python
work = df.copy()
missing_region = work["region"].isna().sum()
work["units"] = work["units"].fillna(round(work["units"].mean()))
work = work.dropna()
final_rows = len(work)
```

---
## Chapter 6 — Cleaning: Duplicates, Text & Types

📖 **Theory.** Real data has duplicate rows, inconsistent text casing, and wrong types.
- `.duplicated()` / `.drop_duplicates()` — find/remove exact duplicate rows
- `.str` accessor — vectorized string ops: `.str.lower()`, `.str.strip()`, `.str.contains()`
- `.astype()` — convert types; `pd.to_datetime()` / `pd.to_numeric()` for parsing

In [ ]:
# Our dataset has some duplicate rows and inconsistent region casing (e.g. 'NORTH' vs 'North')
print("duplicate rows:", df.duplicated().sum())
print("unique region values (note the mess):", df["region"].dropna().unique())

clean = df.drop_duplicates().copy()
print("\nrows after dropping duplicates:", len(clean))

# standardize text: make region title-case consistently
clean["region"] = clean["region"].str.title()   # 'NORTH' -> 'North'
print("regions after cleaning:", clean["region"].dropna().unique())

⚠️ **Common trap.** `'North'` and `'NORTH'` are *different* values to Pandas — they'll be
counted as separate groups in a `groupby` until you standardize the casing. Always inspect
`.unique()` on categorical columns.

### ✏️ Your Turn 6.1
On `clean`:
1. Confirm there are now zero duplicate rows
2. Make all `product` names uppercase
3. Check: does the `order_date` column have a datetime dtype? (look at `clean.dtypes`)

In [ ]:
dup_count = None
# uppercase product...
date_dtype = None
print(dup_count, date_dtype)

✅ **Solution**
```python
dup_count = clean.duplicated().sum()          # 0
clean["product"] = clean["product"].str.upper()
date_dtype = clean["order_date"].dtype        # datetime64[ns]
```

---
## Chapter 7 — Creating & Transforming Columns

📖 **Theory.** Derive new columns from existing ones:
- Direct arithmetic: `df["total"] = df["a"] * df["b"]`
- `.apply(func)` — run a Python function per element (flexible but slower)
- `.map({...})` — map values via a dict
- `np.where(cond, x, y)` — vectorized if/else for a new column
- `pd.cut(...)` — bin a numeric column into categories

In [ ]:
clean = df.drop_duplicates().copy()
clean["region"] = clean["region"].str.title()
clean = clean.dropna(subset=["units"])
clean["units"] = clean["units"].astype(int)

# 1. arithmetic column
clean["revenue_recomputed"] = clean["units"] * clean["unit_price"]

# 2. np.where -- flag high-value orders
clean["order_size"] = np.where(clean["revenue"] >= 1000, "large", "small")

# 3. map -- add a simple region-to-zone mapping
zone_map = {"North":"Zone A","South":"Zone B","East":"Zone A","West":"Zone B"}
clean["zone"] = clean["region"].map(zone_map)

# 4. pd.cut -- bin revenue into tiers
clean["revenue_tier"] = pd.cut(clean["revenue"], bins=[0,100,500,2000,100000],
                               labels=["tiny","small","medium","large"])
clean[["product","revenue","order_size","zone","revenue_tier"]].head()

### ✏️ Your Turn 7.1
On `clean`:
1. Create a `discount_price` column = `unit_price` × 0.9
2. Create a `high_units` boolean column that is True when `units` ≥ 5
3. Use `.apply` to create a `product_label` column that is the product name plus " (" + region + ")"
   — e.g. "LAPTOP (North)". Hint: `.apply(lambda row: ..., axis=1)`

In [ ]:
# discount_price...
# high_units...
# product_label via apply(axis=1)...
print(clean[["product","region"]].head())

✅ **Solution**
```python
clean["discount_price"] = clean["unit_price"] * 0.9
clean["high_units"] = clean["units"] >= 5
clean["product_label"] = clean.apply(lambda r: f"{r['product']} ({r['region']})", axis=1)
```

---
## Chapter 8 — Sorting & Ranking

📖 **Theory.**
- `.sort_values(by, ascending)` — sort rows by one or more columns
- `.sort_index()` — sort by the index
- `.rank()` — assign ranks
- `.nlargest(n, col)` / `.nsmallest(n, col)` — fast top/bottom n

In [ ]:
clean = df.drop_duplicates().dropna(subset=["units"]).copy()

# top 5 orders by revenue
print("Top 5 orders by revenue:")
print(clean.sort_values("revenue", ascending=False).head(5)[["product","revenue"]])

# sort by two keys: region ascending, then revenue descending
multi = clean.sort_values(["region","revenue"], ascending=[True, False])

# nlargest is a shortcut for 'top n by a column'
print("\nTop 3 by revenue (nlargest):")
print(clean.nlargest(3, "revenue")[["product","revenue"]])

### ✏️ Your Turn 8.1
1. Sort the data by `units` descending and show the top 3 products & units
2. Get the 5 *smallest* revenue orders using `.nsmallest`
3. Sort by `sales_rep` (A→Z) and then `revenue` (high→low) within each rep

In [ ]:
top_units = None
smallest_orders = None
rep_then_revenue = None
print(top_units)

✅ **Solution**
```python
top_units = clean.sort_values("units", ascending=False).head(3)[["product","units"]]
smallest_orders = clean.nsmallest(5, "revenue")
rep_then_revenue = clean.sort_values(["sales_rep","revenue"], ascending=[True, False])
```

---
## Chapter 9 — GroupBy & Aggregation ⭐ (the heart of analysis)

📖 **Theory.** This is the single most important Pandas skill. **Split** the data into
groups, **apply** a function to each, **combine** the results. It answers every "X by Y"
business question.

- `df.groupby("col")["value"].sum()` — total value per group
- `.agg({...})` or named aggregations — multiple stats at once
- Group by multiple columns for cross-tabulated summaries

🧠 **Mental model.** GroupBy = "for each unique value of this column, compute a summary."
It's SQL's `GROUP BY`.

In [ ]:
clean = df.drop_duplicates().dropna(subset=["units","region"]).copy()
clean["region"] = clean["region"].str.title()

# total revenue per category
print("Revenue by category:")
print(clean.groupby("category")["revenue"].sum().round(2))

# multiple aggregations at once with named aggregation (the cleanest style)
print("\nRich report by region:")
report = clean.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    avg_order_value=("revenue", "mean"),
    order_count=("revenue", "count"),
    total_units=("units", "sum"),
).round(2)
print(report)

In [ ]:
# group by TWO columns -> cross-tabulated summary
print("Revenue by region AND category:")
two_level = clean.groupby(["region","category"])["revenue"].sum().round(2)
print(two_level)

# which single (region, category) combo earns the most?
print("\nTop combo:", two_level.idxmax(), "->", round(two_level.max(), 2))

⚡ **Pro tip.** Named aggregation — `total=("revenue","sum")` — is the most readable way to
build a multi-metric report. The tuple is `(column_to_aggregate, function)`.

### ✏️ Your Turn 9.1
1. Total `units` sold per `product`
2. A report per `sales_rep` with: total revenue, average revenue, and number of orders
3. Average `revenue` for each combination of `region` and `category`
4. Which `product` has the highest total revenue?

In [ ]:
units_per_product = None
rep_report = None
region_cat_avg = None
top_product = None
print(top_product)

✅ **Solution**
```python
units_per_product = clean.groupby("product")["units"].sum()
rep_report = clean.groupby("sales_rep").agg(
    total_revenue=("revenue","sum"),
    avg_revenue=("revenue","mean"),
    orders=("revenue","count"),
)
region_cat_avg = clean.groupby(["region","category"])["revenue"].mean()
top_product = clean.groupby("product")["revenue"].sum().idxmax()
```

---
## Chapter 10 — Merging & Joining Tables

📖 **Theory.** Real data lives in multiple tables. `pd.merge(left, right, on, how)` combines
them on a shared key — exactly like SQL joins:
- `how="inner"` — only matching keys (default)
- `how="left"` — keep all left rows, fill unmatched right with NaN
- `how="right"` / `how="outer"` — mirror / keep everything

🧠 **Mental model.** Pick `how` by asking "which rows must I keep even if there's no match?"
Defaulting to inner can silently drop rows.

In [ ]:
sales = df.drop_duplicates().dropna(subset=["units"]).copy()
customers = pd.read_csv("datasets/customers.csv", parse_dates=["signup_date"])
print("customers table:")
print(customers.head(3))

# LEFT join: attach customer info to every sale (keep all sales even if customer not found)
merged = sales.merge(customers, on="customer_id", how="left")
print("\nmerged shape:", merged.shape)
print(merged[["product","revenue","segment","city"]].head())

In [ ]:
# now we can analyze across both tables: revenue by customer segment
print("Revenue by customer segment:")
print(merged.groupby("segment")["revenue"].sum().round(2))

# and by city
print("\nTop cities by revenue:")
print(merged.groupby("city")["revenue"].sum().sort_values(ascending=False).round(2))

### ✏️ Your Turn 10.1
Using `merged`:
1. What is the total revenue from `Corporate` segment customers?
2. Which `city` has the highest *average* order revenue?
3. Count how many orders came from each `segment`

In [ ]:
corporate_revenue = None
top_city_by_avg = None
orders_per_segment = None
print(corporate_revenue, top_city_by_avg)

✅ **Solution**
```python
corporate_revenue = merged[merged["segment"]=="Corporate"]["revenue"].sum()
top_city_by_avg = merged.groupby("city")["revenue"].mean().idxmax()
orders_per_segment = merged.groupby("segment").size()
```

---
## Chapter 11 — Datetime & Time-Series Resampling

📖 **Theory.** Pandas has powerful date handling. With a datetime column:
- `.dt` accessor: `.dt.year`, `.dt.month`, `.dt.dayofweek`, `.dt.day_name()`
- Set a datetime as index, then `.resample("M").sum()` to roll up by Month/Week/Day/Quarter
- `.rolling(window)` for moving averages

🧠 **Mental model.** Resampling is `groupby` for time — group rows into time buckets.

In [ ]:
weather = pd.read_csv("datasets/weather_2023.csv", parse_dates=["date"])
print(weather.head(3))

# extract date parts
weather["month"] = weather["date"].dt.month
weather["day_name"] = weather["date"].dt.day_name()

# average temperature per month using groupby
print("\nAvg temp by month:")
print(weather.groupby("month")["temp_c"].mean().round(1))

In [ ]:
# resampling: set date as index, then roll up
ts = weather.set_index("date")

monthly_rain = ts["rainfall_mm"].resample("ME").sum()   # total rain per month
print("Monthly rainfall totals:")
print(monthly_rain.round(1))

# 7-day rolling average temperature (smooths daily noise)
ts["temp_7day_avg"] = ts["temp_c"].rolling(window=7).mean()
print("\nrolling avg (last 5 rows):")
print(ts["temp_7day_avg"].tail())

⚠️ **Common trap.** `.resample()` only works when a datetime is the *index* (or you pass
`on="date"`). Forgetting to set the index is the #1 resampling error. (Note: recent Pandas
uses `"ME"` for month-end; older versions use `"M"`.)

### ✏️ Your Turn 11.1
Using `weather`:
1. Add a column for the day of week (`.dt.dayofweek`, where Monday=0)
2. What was the hottest month on average? (highest mean `temp_c`)
3. Resample to get the *maximum* humidity per week (hint: `resample("W")` on the date-indexed frame)

In [ ]:
# day of week...
hottest_month = None
weekly_max_humidity = None
print(hottest_month)

✅ **Solution**
```python
weather["dow"] = weather["date"].dt.dayofweek
hottest_month = weather.groupby(weather["date"].dt.month)["temp_c"].mean().idxmax()
weekly_max_humidity = weather.set_index("date")["humidity_pct"].resample("W").max()
```

---
## Chapter 12 — Pivot Tables & Reshaping

📖 **Theory.** Reshape between "long" and "wide" formats:
- `pd.pivot_table(df, values, index, columns, aggfunc)` — spreadsheet-style pivot
- `.melt()` — wide → long (unpivot)
- `.stack()` / `.unstack()` — move index levels to/from columns

🧠 **Mental model.** A pivot table is a 2D groupby: rows = one grouping key, columns =
another, cells = an aggregate.

In [ ]:
clean = df.drop_duplicates().dropna(subset=["units","region"]).copy()
clean["region"] = clean["region"].str.title()

# pivot: regions as rows, categories as columns, total revenue in cells
pivot = pd.pivot_table(clean, values="revenue", index="region",
                       columns="category", aggfunc="sum", fill_value=0).round(0)
print("Revenue pivot (region x category):")
print(pivot)

### ✏️ Your Turn 12.1
1. Build a pivot table: `sales_rep` as rows, `region` as columns, showing the **count** of orders
2. Build a pivot: `category` as rows, showing the **mean** revenue (no columns arg needed —
   just index + values + aggfunc)

In [ ]:
rep_region_counts = None
category_avg = None
print(rep_region_counts)

✅ **Solution**
```python
rep_region_counts = pd.pivot_table(clean, values="revenue", index="sales_rep",
                                   columns="region", aggfunc="count", fill_value=0)
category_avg = pd.pivot_table(clean, values="revenue", index="category", aggfunc="mean")
```

---
## 🏆 Chapter 13 — Capstone: Full Business Analysis

You're a data analyst. Leadership wants a complete picture of the retail business from the
raw `retail_sales.csv`. Do the full workflow: **load → clean → analyze → answer**.

Work each task before revealing the solution. This mirrors a real analyst's day.

In [ ]:
# Starting point: load fresh
raw = pd.read_csv("datasets/retail_sales.csv", parse_dates=["order_date"])
print("raw shape:", raw.shape)
print("issues to fix -> duplicates:", raw.duplicated().sum(),
      "| missing units:", raw["units"].isna().sum(),
      "| missing region:", raw["region"].isna().sum())

### ✏️ Capstone Tasks

**Part A — Clean the data**
1. Drop duplicate rows
2. Standardize `region` casing to Title Case
3. Fill missing `region` with `"Unknown"`
4. Drop rows still missing `units`, then convert `units` to integer

**Part B — Analyze**
5. What is total company revenue?
6. Top 3 products by total revenue
7. Which region generates the most revenue?
8. Best-performing sales rep (by total revenue)
9. Monthly revenue trend (revenue per month) — which month was best?
10. Average order value (AOV) per customer segment (requires merging `customers.csv`)

In [ ]:
# ---- Part A: Clean ----
data = raw.drop_duplicates().copy()
# TODO: standardize region casing, fill missing region, drop missing units, cast to int

# ---- Part B: Analyze ----
total_revenue = None
top3_products = None
best_region = None
best_rep = None
monthly_revenue = None
best_month = None
# aov_by_segment requires merging customers.csv

print("Fill in the solution below, then run to check your answers.")

✅ **Capstone Solution**
```python
# ---- Part A: Clean ----
data = raw.drop_duplicates().copy()
data["region"] = data["region"].str.title()
data["region"] = data["region"].fillna("Unknown")
data = data.dropna(subset=["units"])
data["units"] = data["units"].astype(int)

# ---- Part B: Analyze ----
# 5. total revenue
total_revenue = data["revenue"].sum()

# 6. top 3 products
top3_products = data.groupby("product")["revenue"].sum().nlargest(3)

# 7. best region
best_region = data.groupby("region")["revenue"].sum().idxmax()

# 8. best sales rep
best_rep = data.groupby("sales_rep")["revenue"].sum().idxmax()

# 9. monthly revenue trend
monthly_revenue = data.set_index("order_date")["revenue"].resample("ME").sum()
best_month = monthly_revenue.idxmax()

# 10. AOV per segment (merge with customers)
customers = pd.read_csv("datasets/customers.csv")
merged = data.merge(customers, on="customer_id", how="left")
aov_by_segment = merged.groupby("segment")["revenue"].mean().round(2)

print(f"Total revenue: ${total_revenue:,.2f}")
print(f"\nTop 3 products:\n{top3_products}")
print(f"\nBest region: {best_region}")
print(f"Best sales rep: {best_rep}")
print(f"Best month: {best_month.strftime('%B %Y')}")
print(f"\nAOV by segment:\n{aov_by_segment}")
```

🎉 **Congratulations — you've reached Pandas mastery!** You can now load messy real-world
data, clean it, and answer real business questions with groupby, merges, time series, and
pivots. This is exactly what data analysts and ML engineers do every day.

---
### 📌 Function Quick-Reference (everything used in this lab)
**Load/inspect:** `read_csv, head, tail, info, describe, shape, columns, dtypes, isna, notna`
**Select:** `df["col"], df[["a","b"]], .loc, .iloc`
**Filter:** boolean masks with `& | ~`, `.query`, `.isin`
**Missing data:** `isna, dropna, fillna`
**Clean:** `duplicated, drop_duplicates, .str.lower/.upper/.title/.strip/.contains, astype, to_datetime`
**New columns:** arithmetic, `apply, map, np.where, pd.cut`
**Sort/rank:** `sort_values, sort_index, rank, nlargest, nsmallest`
**GroupBy:** `groupby, agg (named aggregation), size, idxmax/idxmin`
**Merge:** `merge (how=inner/left/right/outer)`
**Datetime:** `.dt.year/.month/.dayofweek/.day_name, set_index, resample, rolling`
**Reshape:** `pivot_table, melt, stack, unstack`
